In [3]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def generar_explicabilidad_shap_xgb(target_name):
    """
    Pipeline unificado de explicabilidad SHAP para producción.
    Calcula: SHAP crudo, Porcentajes, Paneles, Direcciones Numéricas/Categóricas 
    para la clase más crítica y reporte de diferencias Global vs Oncológico.
    """
    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DINÁMICA POR TARGET (CLASE CRÍTICA)
    # -------------------------------------------------------------------------
    if target_name == 'SEVERIDAD':
        idx_clase_alta = 3  # Clases: 0, 1, 2, 3
        nombre_efecto_str = 'Severidad Alta (Clase 3)'
    elif target_name == 'CONSUMO_RECURSOS':
        idx_clase_alta = 2  # Clases: 0, 1, 2
        nombre_efecto_str = 'Consumo Alto (Clase 2)'
    else:
        print("Target no reconocido. Ajustar índices.")
        return

    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS Y PARÁMETROS
    # -------------------------------------------------------------------------
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    dir_base_resultados = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}"
    
    nombre_modelo = f"Modelo_Optimo_XGBoost_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    print("="*80)
    print(f"INICIANDO FASE 5 UNIFICADA: SHAP - TARGET: {target_name}")
    print(f"Foco clínico de análisis direccional: {nombre_efecto_str}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo óptimo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo óptimo pre-entrenado desde: {nombre_modelo}...")
    with open(ruta_modelo, 'rb') as f:
        modelo_xgb = pickle.load(f)
        
    print("-> Inicializando SHAP TreeExplainer nativo...")
    explainer = shap.TreeExplainer(modelo_xgb)
    features = modelo_xgb.get_booster().feature_names
    
    # -------------------------------------------------------------------------
    # FUNCIÓN INTERNA DE PROCESAMIENTO
    # -------------------------------------------------------------------------
    def procesar_enfoque_shap(df_origen, tipo_enfoque, nombre_carpeta_sub):
        print(f"\n--- Procesando enfoque: {tipo_enfoque.upper()} (Datos: {len(df_origen)}) ---")
        
        dir_sub_enfoque = os.path.join(dir_base_resultados, nombre_carpeta_sub)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        X_shap = df_origen[features].astype('float32')
        del df_origen; gc.collect()
        
        inicio_time = time.time()
        
        # Bloques SHAP
        batch_size = 10000
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            shap_obj = explainer(batch)
            resultados_list.append(shap_obj.values)
            del batch, shap_obj; gc.collect()
            
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO ONCOLÓGICO DE CONSTANTES ---
        if "onco" in nombre_carpeta_sub.lower():
            varianzas = X_shap.var()
            cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
            if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
                cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
                
            if cols_a_eliminar:
                idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
                X_shap = X_shap.drop(columns=cols_a_eliminar)
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                print(f"      FILTRO: Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        n_clases = matriz_shap.shape[2] if len(matriz_shap.shape) == 3 else 1
        sufijo_archivo = "GLOBAL" if "global" in nombre_carpeta_sub.lower() else "ONCO"
        
        # 1. Guardar respaldo (.npy)
        ruta_npy = os.path.join(dir_sub_enfoque, f"BACKUP_MATRIZ_{target_name}_{sufijo_archivo}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # 2. Exportar CSV Numérico Absoluto
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        if n_clases > 1:
            impacto_total = shap_abs.sum(axis=1)
            columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
            df_shap_imp['Impacto_Total'] = impacto_total
        else:
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=['Impacto_Total'])
            
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')

        # 3. Exportar CSV Porcentual
        print("   -> Generando matriz de importancias porcentuales...")
        df_shap_porcentajes = df_shap_imp.copy()
        cols_num_pct = df_shap_porcentajes.select_dtypes(include=['number']).columns
        for col in cols_num_pct:
            suma_total = df_shap_porcentajes[col].sum()
            if suma_total > 0:
                df_shap_porcentajes[col] = (df_shap_porcentajes[col] / suma_total) * 100
        
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Preparar retorno para el análisis de diferencias
        df_retorno = df_shap_porcentajes.reset_index().rename(columns={'index': 'Variable'})
        
        # 4. Gráficos Summary Plots
        plt.figure(figsize=(12, 8))
        df_top20 = df_shap_imp.head(20)
        df_top20 = df_top20.drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='viridis', ax=plt.gca())
        plt.title(f'Top 20 Variables SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()
        
        plt.figure(figsize=(12, 8))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top20_cat = df_shap_imp.loc[vars_cat_ohe].head(20)
        df_top20_cat = df_top20_cat.drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20_cat.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='plasma', ax=plt.gca())
        plt.title(f'Top 20 Variables Categóricas SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()

        # Aislar matriz de la clase de mayor riesgo
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta]
        
        # 5. Análisis Direccional: Variables Numéricas (Cuartiles)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables numéricas...")
        rangos_direccionales = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try: bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except: bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes = indices_rango.sum()
                    if n_pacientes > 0:
                        promedio_crudo = matriz_clase_alta[indices_rango, idx_var].mean()
                        if promedio_crudo > 0: efecto = "Aumenta probabilidad (+)"
                        elif promedio_crudo < 0: efecto = "Disminuye probabilidad (-)"
                        else: efecto = "Neutral"

                        rangos_direccionales.append({
                            "Variable": v_num, "Rango": str(rango), "N_Pacientes": n_pacientes,
                            f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo, "Efecto_Clinico": efecto
                        })
                        
        df_rangos_num = pd.DataFrame(rangos_direccionales)
        df_rangos_num.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Numericas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 6. Análisis Direccional: Variables Categóricas (OHE 0 vs 1)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables categóricas (OHE)...")
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    if promedio_crudo > 0: efecto = "Aumenta probabilidad (+)"
                    elif promedio_crudo < 0: efecto = "Disminuye probabilidad (-)"
                    else: efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 7. Paneles de Dependencia (Top 20)
        print(f"   -> Generando paneles de dependencia en carpeta...")
        top_20_vars = df_shap_imp.head(20).index.tolist()
        for var in top_20_vars:
            if var in X_shap.columns:
                fig, axes = plt.subplots(1, n_clases, figsize=(5 * n_clases, 4.5))
                for clase in range(n_clases):
                    valores_sh_clase = matriz_shap[:, :, clase]
                    shap.dependence_plot(var, valores_sh_clase, X_shap, interaction_index=None, ax=axes[clase], show=False)
                    axes[clase].set_title(f'Impacto en clase {clase}', fontsize=10)
                
                fig.suptitle(f'Dependence Plot: {var} ({target_name} - {sufijo_archivo})', fontsize=12, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        print(f"   Liberando memoria asignada al enfoque {tipo_enfoque}...")
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes
        gc.collect()
        
        return df_retorno

    # -------------------------------------------------------------------------
    # EJECUCIÓN SECUENCIAL Y REPORTE CRUZADO (DIFERENCIAS)
    # -------------------------------------------------------------------------
    
    print("\n--- PASO A: Cargando datos para análisis global ---")
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)
    
    n_onco_total = len(df_onco_test)
    n_control_needed = 200000 - n_onco_total 
    
    proporciones_control = df_control_test[target_name].value_counts(normalize=True)
    df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(np.round(n_control_needed * proporciones_control[x.name]))), random_state=42)
    )
    
    if len(df_ctrl_sample) != n_control_needed:
        if len(df_ctrl_sample) < n_control_needed:
            dif = n_control_needed - len(df_ctrl_sample)
            extras = df_control_test.drop(df_ctrl_sample.index).sample(n=dif, random_state=42)
            df_ctrl_sample = pd.concat([df_ctrl_sample, extras])
        elif len(df_ctrl_sample) > n_control_needed:
            dif = len(df_ctrl_sample) - n_control_needed
            df_ctrl_sample = df_ctrl_sample.drop(df_ctrl_sample.sample(n=dif, random_state=42).index)
            
    df_test_global = pd.concat([df_onco_test, df_ctrl_sample], ignore_index=True).sample(frac=1, random_state=42)
    del df_control_test, df_ctrl_sample; gc.collect() 
    
    df_global_pct = procesar_enfoque_shap(df_test_global, "Global (Onco + Control)", "Valores SHAP (global)")
    
    print("\n--- PASO B: Cargando datos para análisis oncológico ---")
    df_onco_pct = procesar_enfoque_shap(df_onco_test, "Oncológico Estricto", "Valores SHAP (oncologicos)")
    
    print("\n--- PASO C: Generando reporte comparativo (diferencias Top 20 Onco vs Global) ---")
    df_global_pct['Posicion_Global'] = df_global_pct.index + 1
    df_onco_pct['Posicion_Onco'] = df_onco_pct.index + 1
    
    top_20_onco = df_onco_pct.head(20).copy()
    
    df_comparacion = pd.merge(top_20_onco, df_global_pct, on='Variable', suffixes=('_Onco', '_Global'), how='left')
    df_comparacion['Diferencia_Impacto_Total'] = df_comparacion['Impacto_Total_Onco'] - df_comparacion['Impacto_Total_Global']
    
    col_clase = f"Clase_{idx_clase_alta}"
    col_clase_onco = f"{col_clase}_Onco"
    col_clase_global = f"{col_clase}_Global"
    
    columnas_finales = ['Variable', 'Posicion_Onco', 'Posicion_Global', 'Impacto_Total_Onco', 'Impacto_Total_Global', 'Diferencia_Impacto_Total']
    
    if col_clase_onco in df_comparacion.columns and col_clase_global in df_comparacion.columns:
        nombre_diferencia_clase = f"Diferencia_{col_clase}"
        df_comparacion[nombre_diferencia_clase] = df_comparacion[col_clase_onco] - df_comparacion[col_clase_global]
        columnas_finales.extend([col_clase_onco, col_clase_global, nombre_diferencia_clase])
        
    df_final = df_comparacion[columnas_finales]
    ruta_diferencias = os.path.join(dir_base_resultados, f"Diferencias_SHAP_{target_name}_ONCO_GLOBAL.csv")
    df_final.to_csv(ruta_diferencias, index=False)
    
    print("\n" + "="*80)
    print("PROCESO UNIFICADO FINALIZADO CON ÉXITO")
    print(f"Reportes guardados en: {dir_base_resultados}")
    print("="*80)

# Para ejecutar, descomentar:
# generar_explicabilidad_shap('SEVERIDAD')
# generar_explicabilidad_shap('CONSUMO_RECURSOS')

In [5]:
generar_explicabilidad_shap_xgb('SEVERIDAD')

INICIANDO FASE 5 UNIFICADA: SHAP - TARGET: SEVERIDAD
Foco clínico de análisis direccional: Severidad Alta (Clase 3)
Hora de inicio: 2026-07-15 03:17:49
-> Cargando modelo óptimo pre-entrenado desde: Modelo_Optimo_XGBoost_SEVERIDAD.pkl...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL (ONCO + CONTROL) (Datos: 200000) ---
      -> Procesando bloque 1 de 20...
      -> Procesando bloque 2 de 20...
      -> Procesando bloque 3 de 20...
      -> Procesando bloque 4 de 20...
      -> Procesando bloque 5 de 20...
      -> Procesando bloque 6 de 20...
      -> Procesando bloque 7 de 20...
      -> Procesando bloque 8 de 20...
      -> Procesando bloque 9 de 20...
      -> Procesando bloque 10 de 20...
      -> Procesando bloque 11 de 20...
      -> Procesando bloque 12 de 20...
      -> Procesando bloque 13 de 20...
      -> Procesando bloque 14 de 20...
      -> Procesando bloque 15 de 20...
      -> Procesand

In [4]:
generar_explicabilidad_shap_xgb('CONSUMO_RECURSOS')

INICIANDO FASE 5 UNIFICADA: SHAP - TARGET: CONSUMO_RECURSOS
Foco clínico de análisis direccional: Consumo Alto (Clase 2)
Hora de inicio: 2026-07-15 00:10:27
-> Cargando modelo óptimo pre-entrenado desde: Modelo_Optimo_XGBoost_CONSUMO_RECURSOS.pkl...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL (ONCO + CONTROL) (Datos: 200000) ---
      -> Procesando bloque 1 de 20...
      -> Procesando bloque 2 de 20...
      -> Procesando bloque 3 de 20...
      -> Procesando bloque 4 de 20...
      -> Procesando bloque 5 de 20...
      -> Procesando bloque 6 de 20...
      -> Procesando bloque 7 de 20...
      -> Procesando bloque 8 de 20...
      -> Procesando bloque 9 de 20...
      -> Procesando bloque 10 de 20...
      -> Procesando bloque 11 de 20...
      -> Procesando bloque 12 de 20...
      -> Procesando bloque 13 de 20...
      -> Procesando bloque 14 de 20...
      -> Procesando bloque 15 de 20...
      